In [1]:
# =========================
# 1. Imports
# =========================
import os
import yaml
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import timm

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tqdm import tqdm

print("Imports OK")
print("PyTorch version :", torch.__version__)
print("timm version    :", timm.__version__)

C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK
PyTorch version : 2.11.0+cpu
timm version    : 1.0.26


In [2]:
os.chdir("../..")
print("Project root:", os.getcwd())

Project root: C:\Users\ASUS\Documents\GitHub\Trash-Classification-System


In [3]:
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

SPLITS_CSV   = cfg["data"]["splits_csv"]       # data/splits.csv
SEED         = cfg["data"]["seed"]             # 42
IMG_SIZE     = cfg["data"]["image_size"]       # 224
NUM_CLASSES  = cfg["data"]["num_classes"]      # 6
BATCH_SIZE   = cfg["training"]["batch_size"]   # 32
NUM_EPOCHS   = cfg["training"]["num_epochs"]   # 20
NUM_WORKERS  = cfg["training"]["num_workers"]  # 2
PATIENCE     = cfg["training"]["early_stopping_patience"]  # 5
LR           = cfg["optimizer"]["lr"]          # 0.0001
WEIGHT_DECAY = cfg["optimizer"]["weight_decay"] # 0.01
MIN_LR       = cfg["scheduler"]["min_lr"]      # 0.000001
CLASS_MAP    = cfg["classes"]                  # {Plastic:0, ...}
CKPT_PATH    = Path(cfg["ensemble"]["checkpoint_paths"]["member_3"])

# Reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Config loaded.")
print(f"  Device      : {DEVICE}")
print(f"  Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Batch size  : {BATCH_SIZE}")
print(f"  Epochs      : {NUM_EPOCHS}")
print(f"  LR          : {LR}")
print(f"  Checkpoint  : {CKPT_PATH}")

Config loaded.
  Device      : cpu
  Image size  : 224x224
  Batch size  : 32
  Epochs      : 20
  LR          : 0.0001
  Checkpoint  : models\member_3_mobilenet\checkpoints\best_model.pth


In [4]:
# =========================
# Load Splits
# =========================
df = pd.read_csv(SPLITS_CSV)

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

print(f"Train : {len(train_df)} images")
print(f"Val   : {len(val_df)} images")
print(f"Test  : {len(test_df)} images")
print(f"\nClass distribution in train:")
print(train_df["main_class"].value_counts().to_string())

Train : 10500 images
Val   : 2250 images
Test  : 2250 images

Class distribution in train:
main_class
Plastic    4550
Paper      2100
Metal      1400
Organic    1050
Textile     700
Glass       700


In [5]:
# # =========================
# # 4. Dataset pipeline
# # =========================
# def load_image(path,label):
    
#     img=tf.io.read_file(path)
#     img=tf.image.decode_png(img,channels=3)
#     img=tf.image.resize(img,IMG_SIZE)

#     # MobileNet preprocessing
#     img=preprocess_input(img)

#     return img,label

# def create_dataset(df,training=False):
    
#     paths=df["filepath"].values
#     # Normalize paths: remove backslashes and prepend relative path
#     paths = ["../../" + p.replace("\\", "/") for p in paths]
    
#     labels=df["class_idx"].values
    
#     ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    
#     if training:
#         ds=ds.shuffle(2000,seed=SEED)

#     ds=ds.map(
#         load_image,
#         num_parallel_calls=tf.data.AUTOTUNE
#     )

#     ds=ds.batch(BATCH_SIZE)
#     ds=ds.prefetch(tf.data.AUTOTUNE)
    
#     return ds


# train_ds=create_dataset(train_df,training=True)
# val_ds=create_dataset(val_df)
# test_ds=create_dataset(test_df)

In [21]:
# # Debug: Check first few image paths
# print("First 3 image paths (after normalization):")
# for i, p in enumerate(train_df["filepath"].values[:3]):
#     normalized = "../../" + p.replace("\\", "/")
#     exists = os.path.exists(normalized)
#     print(f"  {normalized} | exists: {exists}")

First 3 image paths (after normalization):
  ../../data/raw/images/aerosol_cans/default/Image_124.png | exists: False
  ../../data/raw/images/shoes/real_world/Image_182.png | exists: False
  ../../data/raw/images/disposable_plastic_cutlery/default/Image_220.png | exists: True


In [22]:
# # =========================
# # 5. Augmentation
# # =========================
# augment=tf.keras.Sequential([
#     layers.RandomFlip("horizontal"),
#     layers.RandomRotation(0.15),
#     layers.RandomZoom(0.15),
#     layers.RandomContrast(0.1)
# ])

In [23]:
# # =========================
# # 6. Class weights
# # =========================
# weights=compute_class_weight(
#     class_weight="balanced",
#     classes=np.unique(train_df.class_idx),
#     y=train_df.class_idx
# )

# class_weights={
#     i:w for i,w in enumerate(weights)
# }

# print(class_weights)

{0: np.float64(0.38461538461538464), 1: np.float64(0.8333333333333334), 2: np.float64(2.5), 3: np.float64(1.25), 4: np.float64(1.6666666666666667), 5: np.float64(2.5)}


In [24]:
# # =========================
# # 7. Build model
# # =========================
# base_model=MobileNetV3Large(
#     include_top=False,
#     weights="imagenet",
#     input_shape=(224,224,3)
# )

# base_model.trainable=False


# inputs=tf.keras.Input(shape=(224,224,3))

# x=augment(inputs)
# x=base_model(x,training=False)

# x=layers.GlobalAveragePooling2D()(x)
# x=layers.BatchNormalization()(x)

# x=layers.Dense(
#     256,
#     activation="relu"
# )(x)

# x=layers.Dropout(0.4)(x)

# outputs=layers.Dense(
#     NUM_CLASSES,
#     activation="softmax"
# )(x)

# model=tf.keras.Model(inputs,outputs)

# model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Large (Functional)   │ (None, 7, 7, 960)      │     2,996,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 960)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 960)            │         3,840 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       246,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,247,750 (12.39 MB)

 Trainable params: 249,478 (974.52 KB)

 Non-trainable params: 2,998,272 (11.44 MB)

In [25]:
# # =========================
# # 8. Compile
# # =========================
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-3),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

In [26]:
# # =========================
# # 9. Callbacks
# # =========================
# callbacks=[
#     EarlyStopping(
#         patience=5,
#         restore_best_weights=True
#     ),

#     ReduceLROnPlateau(
#         factor=0.3,
#         patience=2
#     ),

#     ModelCheckpoint(
#         "best_mobilenet.keras",
#         save_best_only=True
#     )
# ]

In [27]:
# # =========================
# # 10. Initial training
# # =========================
# history1=model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS_STAGE1,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

Epoch 1/15


NotFoundError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
NewRandomAccessFile failed to Create/Open: ../../data/raw/images/plastic_trash_bags/default/Image_123.png : The system cannot find the path specified.
; No such process
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_23263]

In [ ]:
# # =========================
# # 11. Fine tuning
# # unfreeze top layers only
# # =========================
# base_model.trainable=True

# for layer in base_model.layers[:-40]:
#     layer.trainable=False


# model.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-5),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )

# history2=model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS_STAGE2,
#     class_weight=class_weights,
#     callbacks=callbacks
# )

In [ ]:
# # =========================
# # 12. Evaluate
# # =========================
# loss,acc=model.evaluate(test_ds)
# print(f"Test Accuracy: {acc:.4f}")

In [ ]:
# # =========================
# # 13. Classification report
# # =========================
# y_true=np.concatenate(
#     [y for x,y in test_ds],
#     axis=0
# )

# preds=model.predict(test_ds)
# y_pred=np.argmax(preds,axis=1)

# print(
# classification_report(
#     y_true,
#     y_pred
# )
# )

# print(
# confusion_matrix(
#     y_true,
#     y_pred
# )
# )

In [ ]:
# # =========================
# # 14. Save final model
# # =========================
# model.save(
#     "models/mobilenetv3_trash_classifier.keras"
# )

In [6]:
class TrashDataset(Dataset):
    """PyTorch Dataset that reads filepaths from splits.csv.
    Transforms are passed in — training uses augmentation,
    val/test use only resize + normalize.
    """
    def __init__(self, df: pd.DataFrame, transform=None):
        self.filepaths = df["filepath"].values
        self.labels    = df["class_idx"].values
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        # Fix path separators — handles Windows backslashes in splits.csv
        path  = self.filepaths[idx].replace("\\", "/")
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = int(self.labels[idx])
        return image, label

print("TrashDataset defined.")

TrashDataset defined.


In [7]:
# ImageNet mean/std — required for MobileNetV3 pretrained weights from timm
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transforms — augmentation applied HERE, not inside the model
# This ensures val/test never see augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val/test transforms — only resize and normalize, no augmentation
val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print("Transforms defined.")
print("  Train : resize + augment + normalize")
print("  Val   : resize + normalize only")
print("  Test  : resize + normalize only")

Transforms defined.
  Train : resize + augment + normalize
  Val   : resize + normalize only
  Test  : resize + normalize only


In [8]:
train_dataset = TrashDataset(train_df, transform=train_transform)
val_dataset   = TrashDataset(val_df,   transform=val_transform)
test_dataset  = TrashDataset(test_df,  transform=val_transform)

# WeightedRandomSampler — oversample minority classes so each epoch
# sees a more balanced distribution (config: imbalance.use_sampler = true)
train_labels   = train_df["class_idx"].values
class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[label] for label in train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# shuffle=False when using sampler — sampler handles randomisation
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

Train batches : 329
Val batches   : 71
Test batches  : 71


In [9]:
# Class weights for weighted CrossEntropyLoss
# config: imbalance.strategy = class_weights
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)
class_weights = torch.FloatTensor(weights).to(DEVICE)

print("Class weights (higher = minority class penalised more):")
for cls_name, idx in CLASS_MAP.items():
    print(f"  {cls_name:<10}  idx={idx}  weight={weights[idx]:.4f}")

Class weights (higher = minority class penalised more):
  Plastic     idx=0  weight=0.3846
  Paper       idx=1  weight=0.8333
  Glass       idx=2  weight=2.5000
  Metal       idx=3  weight=1.2500
  Organic     idx=4  weight=1.6667
  Textile     idx=5  weight=2.5000


In [10]:
# Load MobileNetV3-Large from timm with pretrained ImageNet weights
# num_classes=0 removes the original classifier head
# We add our own head below
backbone = timm.create_model(
    "mobilenetv3_large_100",
    pretrained=True,
    num_classes=0,       # remove original head
    global_pool="avg"    # global average pooling
)

# Freeze all backbone layers for Stage 1
for param in backbone.parameters():
    param.requires_grad = False

# Get feature dimension from backbone
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    feat_dim = backbone(dummy).shape[1]
print(f"Backbone feature dim: {feat_dim}")

# Custom classifier head
class MobileNetTrash(nn.Module):
    def __init__(self, backbone, feat_dim, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
            # No softmax here — CrossEntropyLoss expects raw logits
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

model = MobileNetTrash(backbone, feat_dim, NUM_CLASSES).to(DEVICE)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,} / {total:,} total")
print("Backbone frozen for Stage 1.")

C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--timm--mobilenetv3_large_100.ra_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Backbone feature dim: 1280
Trainable params : 332,038 / 4,534,070 total
Backbone frozen for Stage 1.


In [11]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Run one training epoch. Returns avg loss and accuracy."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate on val or test loader. Returns avg loss and accuracy."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="  Eval ", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds       = outputs.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += images.size(0)

    return total_loss / total, correct / total


def save_checkpoint(model, path: Path, epoch: int, val_acc: float):
    """Save model weights + metadata to .pth file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "epoch"        : epoch,
        "val_acc"      : val_acc,
        "model_state"  : model.state_dict(),
        "model_name"   : "mobilenetv3_large_100",
        "num_classes"  : NUM_CLASSES,
    }, path)
    print(f"  Checkpoint saved → {path}  (val_acc={val_acc:.4f})")


print("Training helpers defined.")

Training helpers defined.


In [ ]:
# ── Stage 1: Train classifier head only ───────────────────────────────────────
# Backbone is frozen — only head parameters get gradients
# Higher LR is fine here since we're training from scratch on the head

EPOCHS_STAGE1 = 10
LR_STAGE1     = 1e-3    # higher LR for head-only training

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer1 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE1, weight_decay=WEIGHT_DECAY
)
scheduler1 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer1, T_max=EPOCHS_STAGE1, eta_min=MIN_LR
)

best_val_acc   = 0.0
patience_count = 0
history        = {"train_loss": [], "train_acc": [],
                  "val_loss":   [], "val_acc":   []}

print("=" * 55)
print("  Stage 1 — Head only training")
print("=" * 55)

for epoch in range(1, EPOCHS_STAGE1 + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer1, criterion, DEVICE
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler1.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Epoch {epoch:>2}/{EPOCHS_STAGE1}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    # Save best checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, CKPT_PATH, epoch, val_acc)
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}.")
            break

print(f"\nStage 1 complete. Best val_acc = {best_val_acc:.4f}")

  Stage 1 — Head only training


  Train:   0%|                                                                                 | 0/329 [00:00<?, ?it/s]C:\Users\ASUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# ── Stage 2: Unfreeze top layers and fine-tune ─────────────────────────────────
# Unfreeze the last 3 blocks of the backbone
# Use much lower LR to avoid destroying pretrained features

EPOCHS_STAGE2 = 10
LR_STAGE2     = 1e-5   # very low LR for fine-tuning

# Unfreeze all backbone params first
for param in model.backbone.parameters():
    param.requires_grad = True

# Then re-freeze everything except the last 3 named blocks
# timm MobileNetV3 block names: blocks.0 ... blocks.6
UNFREEZE_BLOCKS = ["blocks.4", "blocks.5", "blocks.6", "conv_head", "head"]
for name, param in model.backbone.named_parameters():
    if not any(block in name for block in UNFREEZE_BLOCKS):
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params after unfreeze: {trainable:,}")

optimizer2 = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_STAGE2, weight_decay=WEIGHT_DECAY
)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=EPOCHS_STAGE2, eta_min=MIN_LR
)

patience_count = 0

print("=" * 55)
print("  Stage 2 — Fine-tuning top blocks")
print("=" * 55)

for epoch in range(1, EPOCHS_STAGE2 + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer2, criterion, DEVICE
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)
    scheduler2.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Epoch {epoch:>2}/{EPOCHS_STAGE2}  "
          f"train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  "
          f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(model, CKPT_PATH, epoch, val_acc)
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"  Early stopping at epoch {epoch}.")
            break

print(f"\nStage 2 complete. Best val_acc = {best_val_acc:.4f}")

In [ ]:
# Plot training history — loss and accuracy curves for both stages
total_epochs = len(history["train_loss"])
epochs_range = range(1, total_epochs + 1)
stage2_start = EPOCHS_STAGE1 + 1   # mark where fine-tuning began

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs_range, history["train_loss"], label="Train loss", color="#378ADD")
axes[0].plot(epochs_range, history["val_loss"],   label="Val loss",   color="#D85A30")
axes[0].axvline(stage2_start, color="gray", linestyle="--",
                linewidth=1, label="Stage 2 start")
axes[0].set_title("Loss", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
sns.despine(ax=axes[0])

# Accuracy
axes[1].plot(epochs_range, history["train_acc"], label="Train acc", color="#378ADD")
axes[1].plot(epochs_range, history["val_acc"],   label="Val acc",   color="#D85A30")
axes[1].axvline(stage2_start, color="gray", linestyle="--",
                linewidth=1, label="Stage 2 start")
axes[1].set_title("Accuracy", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()
sns.despine(ax=axes[1])

fig.suptitle("MobileNetV3-Large — Training History",
             fontsize=13, fontweight="bold")
fig.tight_layout()

plot_path = Path("models/member_3_mobilenet/training_history.png")
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Training history saved → {plot_path}")

In [ ]:
# Load best checkpoint before evaluating on test set
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
print(f"Loaded best checkpoint  (epoch={checkpoint['epoch']},"
      f" val_acc={checkpoint['val_acc']:.4f})")

test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

In [ ]:
# Collect all predictions and true labels from test set
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Collecting predictions"):
        images  = images.to(DEVICE)
        outputs = model(images)
        preds   = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

class_names = list(CLASS_MAP.keys())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    ax=ax
)
ax.set_xlabel("Predicted", fontsize=11)
ax.set_ylabel("True", fontsize=11)
ax.set_title("MobileNetV3-Large — Confusion Matrix (Test Set)",
             fontsize=12, fontweight="bold", pad=10)
fig.tight_layout()

cm_path = Path("models/member_3_mobilenet/confusion_matrix.png")
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Confusion matrix saved → {cm_path}")

print("\nShare best_model.pth with Member 4 via Google Drive for ensemble.")
print(f"Checkpoint location: {CKPT_PATH}")